# AiHealth Guard - Colab Model Training Pipeline

Run this notebook in Google Colab to securely train your machine learning models (Logistic Regression, Random Forest, SVM, XGBoost, Neural Network). 
Once complete, it will automatically save the models as `.pkl` and `.keras` files so you can download them and drop them into your project's `backend/models/` folder for Vercel deployment.

### 1. Install Dependencies

In [ ]:
!pip install imbalanced-learn xgboost shap tensorflow scikit-learn pandas numpy

### 2. Fetch & Prepare Cleveland Dataset

In [ ]:
import pandas as pd
import numpy as np
import os
from sklearn.model_selection import train_test_split

FEATURE_COLUMNS = ['age', 'sex', 'cp', 'trestbps', 'chol', 'fbs', 'restecg', 'thalach', 'exang', 'oldpeak', 'slope', 'ca', 'thal']

print("Ensure 'processed.cleveland.data' is uploaded into the Colab environment.")
print("Loading dataset...")
df = pd.read_csv('processed.cleveland.data', names=FEATURE_COLUMNS + ['target'], na_values='?')
X = df[FEATURE_COLUMNS].copy()
y = df['target']
y = y.apply(lambda val: 1 if val > 0 else 0) # Convert to Binary (0 = No IHD, 1 = IHD)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)
print(f"Training set: {X_train.shape[0]} samples. Test set: {X_test.shape[0]} samples.")

### 3. Preprocessing, SMOTE, and Feature Selection (SF-2)

In [ ]:
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from imblearn.over_sampling import SMOTE
import joblib

# SF-2 Requirements according to PRD
SF_2_FEATURES = ['thalach', 'oldpeak', 'ca', 'thal', 'cp', 'exang', 'age', 'chol', 'trestbps', 'slope']
NUMERICAL_FEATURES = ['age', 'trestbps', 'chol', 'thalach', 'oldpeak']

os.makedirs('models', exist_ok=True)

def preprocess_training_data(X_train, y_train):
    X_train_sf2 = X_train[SF_2_FEATURES].copy()
    
    # Imputation
    for col in SF_2_FEATURES:
        if col in NUMERICAL_FEATURES:
            X_train_sf2[col] = X_train_sf2[col].fillna(X_train_sf2[col].median())
        else:
            X_train_sf2[col] = X_train_sf2[col].fillna(X_train_sf2[col].mode()[0])
            
    # Scaling
    scaler = StandardScaler()
    num_sf2 = [f for f in NUMERICAL_FEATURES if f in SF_2_FEATURES]
    X_train_sf2[num_sf2] = scaler.fit_transform(X_train_sf2[num_sf2])
    
    # Save the scaler for our FastAPI backend
    joblib.dump(scaler, 'models/scaler.pkl')
    
    # Apply SMOTE to address class imbalance
    smote = SMOTE(random_state=42)
    X_res, y_res = smote.fit_resample(X_train_sf2, y_train)
    return X_res, y_res, scaler

X_train_resampled, y_train_resampled, scaler = preprocess_training_data(X_train, y_train)
# Save background dataset for SHAP KernelExplainer (used in SVM/NN)
X_train_resampled[:50].to_csv('models/background.csv', index=False)
print(f"Post-SMOTE train shape: {X_train_resampled.shape[0]} samples")

### 4. Train Models

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.svm import SVC
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout

# 1. Logistic Regression
lr = LogisticRegression(max_iter=1000)
lr.fit(X_train_resampled, y_train_resampled)
joblib.dump(lr, 'models/lr.pkl')

# 2. Random Forest
rf = RandomForestClassifier(n_estimators=100, max_depth=5, random_state=42)
rf.fit(X_train_resampled, y_train_resampled)
joblib.dump(rf, 'models/rf.pkl')

# 3. XGBoost (Primary Model based on AiHealth Guard PRD)
xgb = XGBClassifier(n_estimators=100, learning_rate=0.1, max_depth=3, random_state=42)
xgb.fit(X_train_resampled, y_train_resampled)
joblib.dump(xgb, 'models/xgb.pkl')

# 4. SVM
svm = SVC(kernel='rbf', probability=True, random_state=42)
svm.fit(X_train_resampled, y_train_resampled)
joblib.dump(svm, 'models/svm.pkl')

# 5. Neural Network
nn = Sequential([
    Dense(64, activation='relu', input_dim=X_train_resampled.shape[1]),
    Dropout(0.3),
    Dense(32, activation='relu'),
    Dense(1, activation='sigmoid')
])
nn.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
nn.fit(X_train_resampled, y_train_resampled, epochs=100, batch_size=16, verbose=0)
nn.save('models/nn.keras')

print("All models trained and saved!")

### 5. Evaluate Accuracy

In [ ]:
from sklearn.metrics import accuracy_score, roc_auc_score
import json
# Must preprocess X_test identically (with same scaler pipeline)
X_test_sf2 = X_test[SF_2_FEATURES].copy()
num_sf2 = [f for f in NUMERICAL_FEATURES if f in SF_2_FEATURES]
X_test_sf2[num_sf2] = scaler.transform(X_test_sf2[num_sf2])

results = {}
models = {'LR': lr, 'RF': rf, 'XGB': xgb, 'SVM': svm}

for name, model in models.items():
    y_pred = model.predict(X_test_sf2)
    y_prob = model.predict_proba(X_test_sf2)[:, 1]
    results[name] = {
        'Accuracy': accuracy_score(y_test, y_pred),
        'AUC': roc_auc_score(y_test, y_prob)
    }

# NN 
nn_prob = nn.predict(X_test_sf2).flatten()
results['NN'] = {
    'Accuracy': accuracy_score(y_test, (nn_prob > 0.5).astype(int)),
    'AUC': roc_auc_score(y_test, nn_prob)
}

print("\n--- Model Performace ---")
for name, metrics in results.items():
    print(f"{name} - Accuracy: {metrics['Accuracy']*100:.2f}%, AUC: {metrics['AUC']:.4f}")
    
with open('models/metrics.json', 'w') as f:
    json.dump(results, f, indent=4)

### 6. Download the generated files

Click the folder icon on the left sidebar in Colab to view the files.
Download the `models/` directory contents (`lr.pkl`, `rf.pkl`, `xgb.pkl`, `svm.pkl`, `nn.keras`, `scaler.pkl`, `metrics.json`, `background.csv`) and place them cleanly inside your project at `backend/models/`.
